In [1]:
using CMPSExcitations

In [2]:
# canonical basis
function projection_matrix1(D, R)
    Dr, M = eigen(R)

    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix2(D, R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    Dr, M = eigen(R)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        dD1 = view(e, 1:D)
        dD2 = view(e, D+1:2*D)
        X = zeros(D, D)

        k = 2D + 1
        for i in 1:D, j in 1:D
            if i != j
                X[i, j] = e[k] # sets the one hot vector
                k += 1
            end
        end

        Dr = Diagonal(Dr)
        W1 = M * ((X * Dr - Dr * X) + Diagonal(dD1)) / M
        W2 = M * ((X * Dr - Dr * X) + Diagonal(dD2)) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix3(D, R)
    Dr, M = eigen(R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E - M * Diagonal(F) / M
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

excitation_matrix (generic function with 1 method)

In [19]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
# Hcoupled(c, μ) = ∫(
#     (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
#      ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
#      2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁ + c * (ψ̂₂') * (ψ̂₁') * ψ̂₁ * ψ̂₂), (-Inf, +Inf));

In [67]:
c, μ = 10., 5.
tol = 1e-10

Ds = [4, 8]
D = maximum(Ds)

HLL = Hsingle(c, μ)
@time stateLL = find_groundstate(Ds, HLL, YangGaudinCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

# -----
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
HCLL = Hcoupled(c, μ)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\nParticle density: ", expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[], "\nDensity imbalance: ", expval(ψ̂₁' * ψ̂₁ - ψ̂₂' * ψ̂₂, stateCLL)[])

Optimizing D=4


┌ Info: YangGaudinCMPS ground state: initialization with e = 67.373419150670
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 235 iterations: f = -2.734747817523, ‖∇f‖ = 9.7839e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  0.342558 seconds (1.09 M allocations: 48.850 MiB, 2.13% gc time)
---------------
Optimizing D=8


┌ Info: YangGaudinCMPS ground state: converged after 236 iterations: e = -2.734747817523, ‖∇e‖ = 9.7839e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118
┌ Info: YangGaudinCMPS ground state: initialization with e = -2.734747817253
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 352 iterations: f = -2.761265509087, ‖∇f‖ = 9.9120e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 8 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  6.193256 seconds (3.45 M allocations: 317.556 MiB, 0.63% gc time)
---------------
  6.536109 seconds (4.54 M allocations: 366.415 MiB, 0.71% gc time)
Energy density: -2.7612655090871754
 Particle density: 0.436477215379957
 Order parameter: -0.36173976585101364
Energy density: -2.7612655090871203
Particle density: 0.8729544307599033
Density imbalance: 0.0


┌ Info: YangGaudinCMPS ground state: converged after 353 iterations: e = -2.761265509087, ‖∇e‖ = 9.9120e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118


In [68]:
function leftcanonical(state, cmps=false)
    leftgauge!(state)
    r = rightenv(state)[1][]
    D, U = eigen(r)
    Q = U \ state.Q[] * U
    R = U \ state.Rs[1][] * U
    return (cmps) ? InfiniteCMPS(Constant(Q), (Constant(R), Constant(R))) : (Q, R)
end

function left2centre_matrix(ρR)
    D = size(ρR, 1)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)
    factor = sqrt(ρR) # generally can be any C such that CC' = r

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        X1, X2 = W1 * factor, W2 * factor
        M[:, j] = vcat(vec(X1), vec(X2))
    end

    return M
end

left2centre_matrix (generic function with 1 method)

In [69]:
# common setup
# stateCLL = leftcanonical(stateCLL, true)
p = 0 # momentum
R = stateCLL.Rs[1][]
D = size(R, 1) # R = MDᵣ/M
space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
H = excitation_matrix(excitation_operator(HCLL, space), D);

In [70]:
ρR = rightenv(stateCLL)[1][]
ρR ./= tr(ρR)
B = left2centre_matrix(ρR);

In [71]:
P = projection_matrix1(D, R)
vals, vecs = eigen(P' * B' * H * B * P, P' * B' * B * P)
println(real.(vals[1:5]))

[0.30085311897079914, 1.2896735443446246, 2.4820048074230954, 4.046011220627407, 4.364015654359483]


In [72]:
P = Matrix(qr(projection_matrix1(D, R)).Q)
vals, vecs = eigen(P' * B' * H * B * P, P' * B' * B * P)
println(real.(vals[1:5]))

[0.30085311897104955, 1.2896735443439469, 2.4820048074233143, 4.046011220627464, 4.364015654359332]


In [66]:
P = projection_matrix2(D, R)
vals, vecs = eigen(P' * B' * H * B * P, P' * B' * B * P)
println(real.(vals[1:5]))

[0.30085311891157285, 1.289673545771023, 2.482004807978062, 4.046011220135085, 5.134961038612046]


In [33]:
P = Matrix(qr(projection_matrix2(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * B' * H * B * P, P' * B' * B * P)
println(real.(vals[1:5]))

[0.3008531190552109, 1.2896735443295302, 2.4820048072728293, 4.046011220431114, 5.134961038730958]


In [11]:
P = projection_matrix3(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[0.9372912379356917, 3.5432426545474414, 7.656357133286821, 9.294323418024325, 11.02462160536856]


In [12]:
P = Matrix(qr(projection_matrix3(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.9372912379356776, 3.5432426545474454, 7.656357133286828, 9.294323418024316, 11.024621605368576]


QR instead of geneigsolve seems to be consistently equivalent as expected. Different parametrization seem to agree as well. However, the same parametrization gives different results based on the ground state which presumably only varies by gauge.. ??? Specifically the projector seems to be the issue.